# conformance_engine\nGenerated from 02_processing/conformance_engine.py for Databricks notebook import.\n

In [ ]:
"""Conformance engine driven by mapping metadata."""\n\nfrom __future__ import annotations\n\nimport warnings\n\n# Technical columns added by the framework that should never leak into silver.\n_FRAMEWORK_COLS = frozenset({\n    "source_system", "source_entity", "load_id", "ingest_ts",\n    "bronze_dq_status", "bronze_dq_failed_check",\n    "dq_status", "dq_failed_rule",\n})\n\n\ndef apply_column_mappings(df, mapping_rows: list[dict]):\n    """Apply transform expressions from mapping metadata and return a conformance DataFrame.\n\n    Each active mapping row must have:\n    - ``transform_expression``: a valid Spark SQL expression (e.g. ``CAST(src_col AS STRING)``)\n    - ``conformance_column``:   the output column name in the conformance/silver table\n\n    If no valid mapping rows are present, the function returns the input DataFrame\n    minus all known framework technical columns, with a warning.  This prevents\n    bronze metadata columns from silently appearing in silver tables when mappings\n    are accidentally empty.\n\n    Raises ``ValueError`` if the DataFrame select fails due to a bad expression in\n    the mapping metadata — this surfaces CSV configuration errors as clear messages\n    rather than cryptic Spark AnalysisExceptions.\n    """\n    from pyspark.sql import functions as F\n\n    expressions = []\n    for row in mapping_rows:\n        expr_text = row.get("transform_expression")\n        output_col = row.get("conformance_column")\n        if expr_text and output_col:\n            expressions.append(F.expr(expr_text).alias(output_col))\n\n    if not expressions:\n        warnings.warn(\n            "apply_column_mappings: no valid mapping rows found. "\n            "Returning input DataFrame with framework technical columns removed. "\n            "Verify that column_mapping.csv has active rows for this source entity.",\n            stacklevel=2,\n        )\n        passthrough_cols = [c for c in df.columns if c not in _FRAMEWORK_COLS]\n        return df.select(*passthrough_cols) if passthrough_cols else df\n\n    try:\n        return df.select(*expressions)\n    except Exception as exc:\n        raise ValueError(\n            f"conformance_engine: DataFrame select failed — one or more transform_expression "\n            f"values in column_mapping.csv are invalid. Original error: {exc}"\n        ) from exc\n\n\ndef write_conformance(df, table_name: str):\n    df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(table_name)\n